# IRB Copilot — retrieval & RAG experiments

Proves the Phase 3 pipeline (`app.retrieval`, `app.rag`) against the ingested corpus.

Prerequisites:
- select the **IRB Copilot (uv)** kernel;
- ingestion has run (`uv run python -m ingestion`) so Qdrant + `data/bm25_index/` exist;
- `OPENAI_API_KEY` set in `.env` for the vector/hybrid modes and the RAG answer;
- `uv sync --extra rerank` for the `hybrid_rerank` mode.

In [1]:
# Make the repo root importable regardless of where the kernel starts.
import pathlib
import sys

_root = pathlib.Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

from app.config import get_settings
from app.retrieval import Retriever

settings = get_settings()
retriever = Retriever(settings)
question = "What does the EBA require regarding margin of conservatism in LGD estimation?"
print("retrieval_mode =", settings.retrieval_mode, "| prompt_version =", settings.prompt_version)

retrieval_mode = hybrid | prompt_version = v2


## BM25 (lexical — needs no API key)

In [2]:
for h in retriever.search(question, mode="bm25", top_k=5):
    print(f"[{h.score:5.2f}] {h.chunk.doc_id} para {h.chunk.para_ids} p{h.chunk.pages}")
    print("   ", h.chunk.text[:160].strip(), "\n")

[ 5.98] ebagl_2016_07 para ['70'] p[34]
    Where  the  assessment  of  paragraph  67  identifies  differences  in  the  definition  of  default which the process of paragraph 68 reveals to be non-negligi 

[ 5.78] ebagl_2017_16 para ['221'] p[102]
    In addition, it was not clear to respondents why the downturn component was linked to EL BE estimation if this component was to be added only for the LGD in-def 

[ 5.09] ebagl_2017_16 para ['221'] p[102]
    In addition, in order to avoid unwarranted variability the add-on has been specified as a fixed value rather than a floor. With regard to the level of the add-o 

[ 5.03] ebagl_2017_16 para ['221'] p[102]
    The EBA considered this aspect and decided that no specific rules should be specified for low default  portfolios,  as  in  this  case  the  minimum  requiremen 

[ 4.93] ebagl_2017_16 para ['41', '42'] p[59]
    In relation to the requirement that institutions should add a margin of conservatism ('MoC') that is related to the 

/Users/inigo_ocariz/src/llm-zoomcamp-2026/irb-copilot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Compare all four retrieval modes

`vector`/`hybrid` need `OPENAI_API_KEY`; `hybrid_rerank` also needs the `rerank` extra.

In [3]:
for mode in ["bm25", "vector", "hybrid", "hybrid_rerank"]:
    print("###", mode)
    try:
        for h in retriever.search(question, mode=mode, top_k=3):
            print(f"  [{h.score:.3f}] {h.chunk.doc_id} para {h.chunk.para_ids}")
    except Exception as exc:  # missing key / rerank extra
        print("  skipped:", exc)

### bm25
  [5.984] ebagl_2016_07 para ['70']
  [5.779] ebagl_2017_16 para ['221']
  [5.087] ebagl_2017_16 para ['221']
### vector
  [0.674] ecb_gim_2024 para ['189']
  [0.656] ecb_gim_2024 para ['178']
  [0.654] ecb_gim_2024 para ['193']
### hybrid
  [0.028] ecb_gim_2024 para ['179']
  [0.027] ebagl_2017_16 para ['221']
  [0.027] ecb_gim_2024 para ['146', '147']
### hybrid_rerank


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19865.10it/s]


  [0.051] ebagl_2017_16 para ['221']
  [0.044] bcbs_d424_irb para ['150']
  [0.026] ecb_gim_2024 para ['189']


## Full RAG answer (rewrite → retrieve → prompt → LLM)

In [4]:
from app.rag import answer

a = answer(question)
print(a.text)
print("\n--- citations ---")
for c in a.citations:
    print(" -", c.text)
print(
    f"\nrewritten: {a.rewritten_query!r}"
    f"\nmodel={a.model} mode={a.retrieval_mode} prompt={a.prompt_version}"
    f"\ntokens_in={a.tokens_in} tokens_out={a.tokens_out}"
    f" cost=${a.cost_usd:.5f} latency={a.latency_ms}ms"
)

The EBA requires institutions to adopt an appropriate margin of conservatism in the estimation of risk parameters, including LGD, particularly when there are non-negligible differences in the definition of default that cannot be adjusted using external data [EBA Guidelines on the application of the definition of default under Article 178 CRR (EBA/GL/2016/07), para. 70]. This margin of conservatism should reflect the materiality of the remaining differences in the definition of default and their possible impact on all risk parameters [EBA Guidelines on the application of the definition of default under Article 178 CRR (EBA/GL/2016/07), para. 70; EBA Guidelines on PD estimation, LGD estimation and treatment of defaulted exposures (EBA/GL/2017/16), para. 41]. Additionally, institutions should implement a framework for quantification, documentation, and monitoring of estimation errors related to the margin of conservatism [EBA Guidelines on PD estimation, LGD estimation and treatment of de

## Source chunks behind the answer

In [5]:
for s in a.chunks_used:
    print(f"[{s.score:.3f}] {s.doc_title} — para {s.para_ids} p{s.pages}")
    print("   ", s.text[:220].strip(), "\n")

[0.033] EBA Guidelines on the application of the definition of default under Article 178 CRR (EBA/GL/2016/07) — para ['70'] p[34]
    Where  the  assessment  of  paragraph  67  identifies  differences  in  the  definition  of  default which the process of paragraph 68 reveals to be non-negligible but not possible to overcome by adjustments in the exter 

[0.032] EBA Guidelines on PD estimation, LGD estimation and treatment of defaulted exposures (EBA/GL/2017/16) — para ['41', '42'] p[59]
    In relation to the requirement that institutions should add a margin of conservatism ('MoC') that is related to the expected range of estimation errors as required by Articles 179(1)(f) and 180(1)(e)  of  Regulation  (EU 

[0.032] EBA Guidelines on PD estimation, LGD estimation and treatment of defaulted exposures (EBA/GL/2017/16) — para ['5'] p[48]
    These  guidelines  specify  the  requirements  for  the  estimation  of  probability  of  default  (PD) and loss given default (LGD), including LGD